In [9]:
import sys
import os
import numpy as np

# Add project root to path to import config
if "/workspace/Drawing2CAD" not in sys.path:
    sys.path.append("/workspace/Drawing2CAD")

from config.macro import CAD_CMD_ARGS_MASK, CAD_EXT_IDX, CAD_LINE_IDX, CAD_ARC_IDX, CAD_CIRCLE_IDX, CAD_N_ARGS_EXT, CAD_N_ARGS_PLANE, CAD_N_ARGS_TRANS, CAD_N_ARGS_EXT_PARAM

def calculate_accuracy(out_vec, gt_vec, tolerance=3):
    # out_vec, gt_vec: (S, 1 + N_ARGS)
    
    pred_cmd = out_vec[:, 0] # (S,)
    gt_cmd = gt_vec[:, 0]
    
    pred_args = out_vec[:, 1:] # (S, N_ARGS)
    gt_args = gt_vec[:, 1:]
    
    # Command Accuracy
    cmd_match = (pred_cmd == gt_cmd)
    cmd_acc = np.mean(cmd_match)
    
    # Argument Accuracy (Tolerance)
    # 1. Check tolerance
    args_close = (np.abs(gt_args - pred_args) <= tolerance)
    
    # 2. Filter: Only consider args where command was predicted correctly
    # We will slice using indices where (gt_cmd == TYPE) AND (cmd_match is True)
    
    # Helper to get filtered indices for a specific command type
    def get_indices(cmd_type):
        return np.where((gt_cmd == cmd_type) & cmd_match)[0]
    
    ext_pos = get_indices(CAD_EXT_IDX)
    line_pos = get_indices(CAD_LINE_IDX)
    arc_pos = get_indices(CAD_ARC_IDX)
    circle_pos = get_indices(CAD_CIRCLE_IDX)
    
    results = {}
    
    # Line: first 2 args
    if len(line_pos) > 0:
        results['line'] = args_close[line_pos][:, :2].flatten().astype(np.int32)
    else:
        results['line'] = np.array([])
        
    # Arc: first 4 args
    if len(arc_pos) > 0:
        results['arc'] = args_close[arc_pos][:, :4].flatten().astype(np.int32)
    else:
        results['arc'] = np.array([])
        
    # Circle: args 0, 1, 4
    if len(circle_pos) > 0:
        results['circle'] = args_close[circle_pos][:, [0, 1, 4]].flatten().astype(np.int32)
    else:
        results['circle'] = np.array([])
        
    # Extrude
    if len(ext_pos) > 0:
        ext_args = args_close[ext_pos][:, -CAD_N_ARGS_EXT:]
        results['plane'] = ext_args[:, :CAD_N_ARGS_PLANE].flatten().astype(np.int32)
        results['trans'] = ext_args[:, CAD_N_ARGS_PLANE:CAD_N_ARGS_PLANE+CAD_N_ARGS_TRANS].flatten().astype(np.int32)
        results['extent'] = ext_args[:, -CAD_N_ARGS_EXT_PARAM].flatten().astype(np.int32)
    else:
        results['plane'] = np.array([])
        results['trans'] = np.array([])
        results['extent'] = np.array([])
    # print(results)
    return cmd_acc, results

In [25]:
import os
import h5py
from tqdm import tqdm

test_results = "/workspace/Drawing2CAD/proj_log/share_decoder_6layer_4x_train/test_results"

total_cmd_acc = 0
all_results = {
    'line': [], 'arc': [], 'circle': [],
    'plane': [], 'trans': [], 'extent': []
}

file_list = os.listdir(test_results)
for h5_file in tqdm(file_list):
    h5_file_path = os.path.join(test_results, h5_file)
    with h5py.File(h5_file_path, 'r') as f:
        out_vec = f['out_vec'][:] # 5x17
        gt_vec = f['gt_vec'][:] # 5x17
    cmd_acc, results = calculate_accuracy(out_vec, gt_vec, tolerance=3)
    total_cmd_acc += cmd_acc
    
    for k, v in results.items():
        if len(v) > 0:
            all_results[k].append(v)

print(f"Total Cmd Accuracy: {total_cmd_acc / len(file_list)}")
print("Argument Accuracies:")
total_args_accuracy = 0
for k, v in all_results.items():
    if len(v) > 0:
        acc = np.mean(np.concatenate(v))
        total_args_accuracy += acc
        print(f"  {k}: {acc}")
    else:
        print(f"  {k}: N/A")
print(f"Total Args Accuracy: {total_args_accuracy / len(all_results)}")

  1%|          | 84/7881 [00:00<00:28, 277.27it/s]

100%|██████████| 7881/7881 [00:27<00:00, 287.61it/s]

Total Cmd Accuracy: 0.8205172235454038
Argument Accuracies:
  line: 0.6969406246668799
  arc: 0.7948058413541321
  circle: 0.9154744873628994
  plane: 0.9417677511997884
  trans: 0.7019328874277293
  extent: 0.6646638703094887
Total Args Accuracy: 0.7859309103868196
